In [7]:
import numpy as np
import time
from tensorflow.keras.datasets import mnist
from cnn_base import LeNet5, cross_entropy_loss
from numba import jit

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

num_samples = 5000
train_images = train_images[:num_samples]
train_labels = train_labels[:num_samples]
test_images = test_images[:num_samples]
test_labels = test_labels[:num_samples]

# Apply numba's JIT for preprocessing (without padding and reshape)
@jit(nopython=True)
def preprocess_image(img):
    img = img.astype(np.float32) / 255.0
    return img

# Process images in a loop instead of list comprehension
def preprocess_images(images):
    processed_images = np.empty((len(images), 32, 32, 1), dtype=np.float32)  # Corrected shape
    for i in range(len(images)):
        img = preprocess_image(images[i])
        img_padded = np.pad(img, ((2, 2), (2, 2)), mode='constant')  # Padding to 32x32
        processed_images[i] = img_padded.reshape(32, 32, 1)  # Reshape to (32, 32, 1)
    return processed_images

# Process images (note: we're not resizing here)
train_images_processed = preprocess_images(train_images)
test_images_processed = preprocess_images(test_images)

# INIT
model = LeNet5(learning_rate=0.01)

# Training loop (do not apply numba to TensorFlow-related code)
num_epochs = 10
train_losses = []
train_accuracies = []

print("Starting training...\n")

def compute_epoch_metrics(model, train_images_processed, train_labels, num_samples):
    epoch_loss = 0.0
    correct = 0
    for i in range(num_samples):
        x = train_images_processed[i]
        y = train_labels[i]
        loss, y_pred = model.train_step(x, y)
        epoch_loss += loss

        pred_label = np.argmax(y_pred)
        if pred_label == y:
            correct += 1
    epoch_loss /= num_samples
    accuracy = (correct / num_samples) * 100.0
    return epoch_loss, accuracy

# Training loop without numba for TensorFlow operations
for epoch in range(num_epochs):
    start_time = time.time()
    epoch_loss, accuracy = compute_epoch_metrics(model, train_images_processed, train_labels, num_samples)
    train_losses.append(epoch_loss)
    train_accuracies.append(accuracy)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Accuracy: {accuracy:.2f}% - Time: {time.time()-start_time:.2f}s")

# Evaluation loop (no need to JIT optimize TensorFlow operations)
test_loss = 0.0
test_correct = 0
predictions = []

def compute_test_metrics(model, test_images_processed, test_labels, num_samples):
    test_loss = 0.0
    test_correct = 0
    predictions = []
    for i in range(num_samples):
        x = test_images_processed[i]
        y = test_labels[i]
        y_pred, _ = model.forward(x)
        loss, _ = cross_entropy_loss(y_pred, y)
        test_loss += loss
        pred_label = np.argmax(y_pred)
        predictions.append(pred_label)
        if pred_label == y:
            test_correct += 1
    return test_loss, test_correct, predictions

test_loss, test_correct, predictions = compute_test_metrics(model, test_images_processed, test_labels, num_samples)

test_loss /= num_samples
test_accuracy = (test_correct / num_samples) * 100.0

print("\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Sample predictions
print("\nSample Predictions (first 10 examples):")
for i in range(10):
    print(f"Sample {i+1}: True Label = {test_labels[i]}, Predicted = {predictions[i]}")


Starting training...



ValueError: negative dimensions are not allowed

In [9]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


Num GPUs Available:  0
